In [17]:
# Import libraries

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle
import warnings
warnings.filterwarnings('ignore')

In [18]:
# LOAD ENGINEERED FEATURES

print("\n LOADING ENGINEERED FEATURES...")
print("-" * 80)

df = pd.read_csv('../data/processed/features_engineered.csv')
df['date'] = pd.to_datetime(df['date'])
df['dob'] = pd.to_datetime(df['dob'])

print(f"✓ Loaded {len(df):,} records for {df['child_id'].nunique():,} children")



 LOADING ENGINEERED FEATURES...
--------------------------------------------------------------------------------
✓ Loaded 310,850 records for 57,684 children


In [19]:
# CRITICAL: CLEAN INVALID VALUES BEFORE PROCESSING

print("\n CLEANING INVALID VALUES...")
print("-" * 80)

# Replace inf with NaN for easier handling
df = df.replace([np.inf, -np.inf], np.nan)

# Check NaN counts BEFORE cleaning
print("NaN counts before cleaning:")
nan_counts = df.isna().sum()
nan_counts = nan_counts[nan_counts > 0]
if len(nan_counts) > 0:
    print(nan_counts)
else:
    print("  No NaN values found")

# Strategy 1: Drop rows with NaN in critical columns
critical_cols = ['height', 'weight', 'age_months', 'cbmi', 'gender_numeric']
before_drop = len(df)
df = df.dropna(subset=critical_cols)
after_drop = len(df)
print(f"\n✓ Dropped {before_drop - after_drop:,} rows with NaN in critical columns")

# Strategy 2: Fill NaN in derived features with 0
derived_cols = ['height_velocity_monthly', 'weight_velocity_monthly', 
                'zlen', 'zwei', 'zwfl', 'zbmi',
                'zlen_change', 'zwei_change', 'zwfl_change', 'zbmi_change']
for col in derived_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

print(f"✓ Filled NaN in derived features with 0")

# Verify no NaN remains
remaining_nan = df.isna().sum().sum()
print(f"✓ Remaining NaN values: {remaining_nan}")

if remaining_nan > 0:
    print("Still have NaN values, filling all with 0...")
    df = df.fillna(0)
    print(f"✓ Final NaN count: {df.isna().sum().sum()}")

print(f"✓ Clean dataset: {len(df):,} records")


 CLEANING INVALID VALUES...
--------------------------------------------------------------------------------
NaN counts before cleaning:
zlen    4531
zwei    4531
zwfl    9883
zbmi    4531
dtype: int64

✓ Dropped 0 rows with NaN in critical columns
✓ Filled NaN in derived features with 0
✓ Remaining NaN values: 0
✓ Clean dataset: 310,850 records


In [20]:
# CREATE SEQUENCE DATA

print("\n CREATING TIME-SERIES SEQUENCES")
print("-" * 80)

MIN_HISTORY = 2
MAX_SEQUENCE_LENGTH = 10
PREDICTION_HORIZON = 1

input_features = [
    'age_months', 'height', 'weight', 'cbmi',
    'zlen', 'zwei', 'zwfl', 'zbmi',
    'gender_numeric', 'height_velocity_monthly',
    'weight_velocity_monthly', 'measurement_number'
]


 CREATING TIME-SERIES SEQUENCES
--------------------------------------------------------------------------------


In [21]:
# CRITICAL FIX: FORCE NUMERIC CONVERSION

print("Forcing all input features to be numeric...")
for col in input_features:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(0)

print(f"Feature types after cleaning:\n{df[input_features].dtypes}")

df = df.sort_values(['child_id', 'date']).reset_index(drop=True)

target_features = ['height', 'weight']

sequences = []
sequence_info = []

print(f"Creating sequences with:")
print(f"  - Minimum history: {MIN_HISTORY} measurements")
print(f"  - Maximum sequence length: {MAX_SEQUENCE_LENGTH}")
print(f"  - Prediction horizon: {PREDICTION_HORIZON} measurement(s)")

for child_id, child_data in df.groupby('child_id'):
    child_data = child_data.sort_values('date').reset_index(drop=True)
    n_measurements = len(child_data)
    
    for i in range(MIN_HISTORY, n_measurements):
        history_length = min(i, MAX_SEQUENCE_LENGTH)
        start_idx = i - history_length
        
        try:
            input_sequence = child_data.iloc[start_idx:i][input_features].values.astype(float)
            target_sequence = child_data.iloc[i][target_features].values.astype(float)
            target_age = float(child_data.iloc[i]['age_months'])
        except ValueError:
            continue

        if np.isnan(input_sequence).any() or np.isnan(target_sequence).any():
            continue 
        
        sequences.append({
            'child_id': child_id,
            'input_sequence': input_sequence,
            'target': target_sequence,
            'target_age': target_age,
            'sequence_length': len(input_sequence),
            'gender': child_data.iloc[0]['gender_numeric']
        })

print(f"✓ Created {len(sequences):,} training sequences")

Forcing all input features to be numeric...
Feature types after cleaning:
age_months                 float64
height                     float64
weight                     float64
cbmi                       float64
zlen                       float64
zwei                       float64
zwfl                       float64
zbmi                       float64
gender_numeric               int64
height_velocity_monthly    float64
weight_velocity_monthly    float64
measurement_number           int64
dtype: object
Creating sequences with:
  - Minimum history: 2 measurements
  - Maximum sequence length: 10
  - Prediction horizon: 1 measurement(s)
✓ Created 195,482 training sequences


In [22]:
# PREPARE ARRAYS FOR MODELING

print("\n PREPARING ARRAYS FOR MODELING")
print("-" * 80)

# Pad sequences to same length (needed for batch processing)
def pad_sequence(seq, max_len, n_features):
    """Pad sequence with zeros to max_len"""
    if len(seq) < max_len:
        padding = np.zeros((max_len - len(seq), n_features))
        return np.vstack([padding, seq])
    return seq

n_features = len(input_features)

# Force dtype=float32 to prevent Object Arrays
X_padded = np.array([
    pad_sequence(s['input_sequence'], MAX_SEQUENCE_LENGTH, n_features)
    for s in sequences
]).astype(np.float32)

y = np.array([s['target'] for s in sequences]).astype(np.float32)
sequence_lengths = np.array([s['sequence_length'] for s in sequences])
target_ages = np.array([s['target_age'] for s in sequences])
child_ids = np.array([s['child_id'] for s in sequences])
genders = np.array([s['gender'] for s in sequences])

print(f"✓ Prepared arrays:")
print(f"  X shape: {X_padded.shape} (samples, timesteps, features)")
print(f"  y shape: {y.shape} (samples, targets)")
print(f"  Sequence lengths: {sequence_lengths.shape}")


 PREPARING ARRAYS FOR MODELING
--------------------------------------------------------------------------------
✓ Prepared arrays:
  X shape: (195482, 10, 12) (samples, timesteps, features)
  y shape: (195482, 2) (samples, targets)
  Sequence lengths: (195482,)


In [23]:
# CRITICAL: FINAL INVALID VALUE CHECK AND FIX

print("\n FINAL INVALID VALUE CHECK...")
print("-" * 80)

print(f"X_padded - NaN count: {np.isnan(X_padded).sum()}")
print(f"X_padded - Inf count: {np.isinf(X_padded).sum()}")
print(f"y - NaN count: {np.isnan(y).sum()}")
print(f"y - Inf count: {np.isinf(y).sum()}")

# Replace any NaN or Inf with safe values
X_padded = np.nan_to_num(X_padded, nan=0.0, posinf=1e6, neginf=-1e6)
y = np.nan_to_num(y, nan=0.0, posinf=1e6, neginf=-1e6)

# Clip extreme values for numerical stability
X_padded = np.clip(X_padded, -1e6, 1e6)
y = np.clip(y, 1.0, 1e6)

print(f"\n✓ After cleaning:")
print(f"  X_padded - NaN: {np.isnan(X_padded).any()}, Inf: {np.isinf(X_padded).any()}")
print(f"  y - NaN: {np.isnan(y).any()}, Inf: {np.isinf(y).any()}")
print(f"  X range: [{X_padded.min():.2f}, {X_padded.max():.2f}]")
print(f"  y range: [{y.min():.2f}, {y.max():.2f}]")

print(f"\n✓ Prepared arrays:")
print(f"  X shape: {X_padded.shape} (samples, timesteps, features)")
print(f"  y shape: {y.shape} (samples, targets)")


 FINAL INVALID VALUE CHECK...
--------------------------------------------------------------------------------
X_padded - NaN count: 0
X_padded - Inf count: 0
y - NaN count: 0
y - Inf count: 0

✓ After cleaning:
  X_padded - NaN: False, Inf: False
  y - NaN: False, Inf: False
  X range: [-30.44, 549.80]
  y range: [1.00, 185.00]

✓ Prepared arrays:
  X shape: (195482, 10, 12) (samples, timesteps, features)
  y shape: (195482, 2) (samples, targets)


In [24]:
# TRAIN/VALIDATION/TEST SPLIT

print("\n SPLITTING DATA")
print("-" * 80)

unique_children = np.unique(child_ids)
n_children = len(unique_children)

# First split: separate test set
train_val_children, test_children = train_test_split(
    unique_children,
    test_size=0.15,
    random_state=42
)

# Second split: separate validation from training
train_children, val_children = train_test_split(
    train_val_children,
    test_size=0.15/0.85,
    random_state=42
)

# Create masks for each split
train_mask = np.isin(child_ids, train_children)
val_mask = np.isin(child_ids, val_children)
test_mask = np.isin(child_ids, test_children)

# Split the data
X_train = X_padded[train_mask]
y_train = y[train_mask]
lengths_train = sequence_lengths[train_mask]

X_val = X_padded[val_mask]
y_val = y[val_mask]
lengths_val = sequence_lengths[val_mask]

X_test = X_padded[test_mask]
y_test = y[test_mask]
lengths_test = sequence_lengths[test_mask]

print(f"✓ Data split completed:")
print(f"\nChildren split:")
print(f"  Train: {len(train_children):,} children ({len(train_children)/n_children*100:.1f}%)")
print(f"  Validation: {len(val_children):,} children ({len(val_children)/n_children*100:.1f}%)")
print(f"  Test: {len(test_children):,} children ({len(test_children)/n_children*100:.1f}%)")

print(f"\nSequences split:")
print(f"  Train: {len(X_train):,} sequences ({len(X_train)/len(X_padded)*100:.1f}%)")
print(f"  Validation: {len(X_val):,} sequences ({len(X_val)/len(X_padded)*100:.1f}%)")
print(f"  Test: {len(X_test):,} sequences ({len(X_test)/len(X_padded)*100:.1f}%)")


 SPLITTING DATA
--------------------------------------------------------------------------------
✓ Data split completed:

Children split:
  Train: 34,373 children (70.0%)
  Validation: 7,366 children (15.0%)
  Test: 7,366 children (15.0%)

Sequences split:
  Train: 137,041 sequences (70.1%)
  Validation: 29,230 sequences (15.0%)
  Test: 29,211 sequences (14.9%)


In [25]:
# FEATURE SCALING

print("\n5. SCALING FEATURES")
print("-" * 80)

# Reshape for scaling
X_train_reshaped = X_train.reshape(-1, n_features)

# Fit scaler on training data
scaler = StandardScaler()
scaler.fit(X_train_reshaped)

# Transform all sets
def scale_sequences(X, scaler):
    """Scale 3D sequence data"""
    original_shape = X.shape
    X_reshaped = X.reshape(-1, n_features)
    X_scaled = scaler.transform(X_reshaped)
    return X_scaled.reshape(original_shape)

X_train_scaled = scale_sequences(X_train, scaler)
X_val_scaled = scale_sequences(X_val, scaler)
X_test_scaled = scale_sequences(X_test, scaler)

print("✓ Features scaled using StandardScaler")
print("  - Fitted on training data only")
print("  - Applied to train, validation, and test sets")

# Save scaler
with open('../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✓ Saved scaler to: ../data/processed/scaler.pkl")



5. SCALING FEATURES
--------------------------------------------------------------------------------
✓ Features scaled using StandardScaler
  - Fitted on training data only
  - Applied to train, validation, and test sets
✓ Saved scaler to: ../data/processed/scaler.pkl


In [26]:
# SAVE PREPARED DATA

print("\n6. SAVING PREPARED DATA")
print("-" * 80)

# Save as numpy arrays
np.savez_compressed('../data/processed/train_data.npz',
                    X=X_train_scaled,
                    y=y_train,
                    lengths=lengths_train)

np.savez_compressed('../data/processed/val_data.npz',
                    X=X_val_scaled,
                    y=y_val,
                    lengths=lengths_val)

np.savez_compressed('../data/processed/test_data.npz',
                    X=X_test_scaled,
                    y=y_test,
                    lengths=lengths_test)

print("✓ Saved training data to: ../data/processed/train_data.npz")
print("✓ Saved validation data to: ../data/processed/val_data.npz")
print("✓ Saved test data to: ../data/processed/test_data.npz")

# Save metadata
metadata = {
    'input_features': input_features,
    'target_features': target_features,
    'n_features': n_features,
    'n_targets': len(target_features),
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'min_history': MIN_HISTORY,
    'train_children': len(train_children),
    'val_children': len(val_children),
    'test_children': len(test_children),
    'train_sequences': len(X_train),
    'val_sequences': len(X_val),
    'test_sequences': len(X_test)
}

with open('../data/processed/metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print("✓ Saved metadata to: ../data/processed/metadata.pkl")


6. SAVING PREPARED DATA
--------------------------------------------------------------------------------
✓ Saved training data to: ../data/processed/train_data.npz
✓ Saved validation data to: ../data/processed/val_data.npz
✓ Saved test data to: ../data/processed/test_data.npz
✓ Saved metadata to: ../data/processed/metadata.pkl


In [27]:
# DATA STATISTICS

print("\n DATA STATISTICS")
print("-" * 80)

print("\nTarget Variable Statistics (Training Set):")
print(f"Height (cm):")
print(f"  Mean: {y_train[:, 0].mean():.2f} ± {y_train[:, 0].std():.2f}")
print(f"  Range: [{y_train[:, 0].min():.1f}, {y_train[:, 0].max():.1f}]")

print(f"\nWeight (kg):")
print(f"  Mean: {y_train[:, 1].mean():.2f} ± {y_train[:, 1].std():.2f}")
print(f"  Range: [{y_train[:, 1].min():.1f}, {y_train[:, 1].max():.1f}]")

print(f"\nSequence Length Distribution:")
unique_lengths, counts = np.unique(lengths_train, return_counts=True)
for length, count in zip(unique_lengths, counts):
    print(f"  Length {length}: {count:,} sequences ({count/len(lengths_train)*100:.1f}%)")


 DATA STATISTICS
--------------------------------------------------------------------------------

Target Variable Statistics (Training Set):
Height (cm):
  Mean: 88.67 ± 12.15
  Range: [40.0, 184.0]

Weight (kg):
  Mean: 12.46 ± 3.37
  Range: [1.0, 100.0]

Sequence Length Distribution:
  Length 2: 34,373 sequences (25.1%)
  Length 3: 29,222 sequences (21.3%)
  Length 4: 24,859 sequences (18.1%)
  Length 5: 19,650 sequences (14.3%)
  Length 6: 14,793 sequences (10.8%)
  Length 7: 9,744 sequences (7.1%)
  Length 8: 3,626 sequences (2.6%)
  Length 9: 773 sequences (0.6%)
  Length 10: 1 sequences (0.0%)


In [28]:
# PREPARATION SUMMARY

print("\n" + "=" * 80)
print("DATA PREPARATION SUMMARY")
print("=" * 80)

print(f"\n✓ Sequences Created: {len(sequences):,}")
print(f"✓ Input Features: {n_features}")
print(f"✓ Target Features: {len(target_features)} (height, weight)")
print(f"✓ Max Sequence Length: {MAX_SEQUENCE_LENGTH}")

print(f"\n✓ Training Set: {len(X_train):,} sequences from {len(train_children):,} children")
print(f"✓ Validation Set: {len(X_val):,} sequences from {len(val_children):,} children")
print(f"✓ Test Set: {len(X_test):,} sequences from {len(test_children):,} children")


DATA PREPARATION SUMMARY

✓ Sequences Created: 195,482
✓ Input Features: 12
✓ Target Features: 2 (height, weight)
✓ Max Sequence Length: 10

✓ Training Set: 137,041 sequences from 34,373 children
✓ Validation Set: 29,230 sequences from 7,366 children
✓ Test Set: 29,211 sequences from 7,366 children
